# PONE-D-26-31931 — Tier A revision analyses (no training)

This notebook only **reads** your existing data, models and result CSVs. It never overwrites them; every output goes to `OUT_DIR`.

| Section | Reviewer comment | Needs GPU? | Rough time on L4 |
|---|---|---|---|
| 0. Setup + config | – | no | 2 min |
| 1. Inventory | all | no | 5–10 min |
| 2. NCT-CRC class verification | R1-7 | no | 5 min |
| 3. BreakHis extra image (7,910 vs 7,909) | R1 minor | no | 10 min |
| 4. Achieved compression ratios | R1-6 | no (CPU) | 20–60 min |
| 5. Re-predict from saved models + check vs original CSVs | R1-3, R2-6 | **yes** | 30–60 min |
| 6. Full metrics + revised statistics | R1-2, R1-3 | no | 5 min |

**Run order:** 0 → 1 → read the inventory → fix anything in `CFG` that it flags → 2 → 3 → 4 → 5 → 6.  
Sections 4 and 5 are resumable: if Colab disconnects, re-run sections 0 and 1, then re-run the section; finished parts are skipped.

**Most important check:** Section 5 prints, for every experiment, whether the re-predicted fold accuracies match your original CSVs. If they don't match, **stop** and send me the output. A mismatch means the fold reconstruction differs from the original run, and the statistics in Section 6 would not be valid.

## 0. Setup

In [1]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run('apt-get -qq install -y libopenjp2-7 > /dev/null', shell=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'glymur', 'scikit-image'])

import re, gc, glob, json, time, hashlib, tempfile, warnings
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, matthews_corrcoef
from scipy import stats
import glymur
from skimage.metrics import peak_signal_noise_ratio as sk_psnr
warnings.filterwarnings('ignore')

print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
print('Glymur', glymur.__version__, '| OpenJPEG', glymur.version.openjpeg_version)

Mounted at /content/drive
TensorFlow 2.21.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Glymur 0.14.8 | OpenJPEG 2.5.0


In [2]:
# ============================================================
# CONFIG — edit paths here if Section 1 reports something missing
# ============================================================
DRIVE = '/content/drive/MyDrive'

CFG = {
    'out_dir': f'{DRIVE}/JPEG2000_Revision_TierA',
    'cr_levels': ['CR1_original', 'CR15', 'CR25', 'CR50', 'CR100', 'CR500', 'CR1000'],
    'cr_value': {'CR15': 15, 'CR25': 25, 'CR50': 50, 'CR100': 100, 'CR500': 500, 'CR1000': 1000},
    'mixed_cr_levels': ['CR1_original', 'CR15', 'CR25', 'CR50', 'CR100'],
    'seed': 42, 'n_folds': 5, 'image_size': (224, 224), 'batch_size': 64,
    'n_boot': 10000,

    # Paths below come from the original notebooks (2/27 BreakHis "A-to-Z" run, BACH run, 6/21 NCT run).
    'datasets': {
        'BreakHis': {
            'loader': 'breakhis', 'cv': 'rotating',
            'data_dir': f'{DRIVE}/BreakHis_Split',
            'model_dir': f'{DRIVE}/BreakHis_PLOS_ONE_FINAL/models',
            'results_dir': f'{DRIVE}/BreakHis_PLOS_ONE_FINAL/results',
            'model_name': 'Inception_DenseNet169_Attention',
        },
        'BACH': {
            'loader': 'bach', 'cv': 'rotating',
            'data_dir': f'{DRIVE}/BACH_PLOS_ONE/BACH_Split',
            'model_dir': f'{DRIVE}/BACH_PLOS_ONE/BACH_PLOS_ONE_FINAL/models',
            'results_dir': f'{DRIVE}/BACH_PLOS_ONE/BACH_PLOS_ONE_FINAL/results',
            'model_name': 'Inception_DenseNet169_Attention',
        },
        'NCT-CRC': {
            'loader': 'nct', 'cv': 'fixed_test',
            'data_dir': f'{DRIVE}/NCT_CRC_PLOS_ONE/NCT_CRC_Split',
            # From 'NCT-CRC JPEG2000 Compression Experiments.ipynb' (6/22 balanced-split run = manuscript numbers)
            'model_dir': f'{DRIVE}/NCT_CRC_PLOS_ONE/NCT_CRC_Results_Final/models',
            'results_dir': f'{DRIVE}/NCT_CRC_PLOS_ONE/NCT_CRC_Results_Final/results',
            'split_file': f'{DRIVE}/NCT_CRC_PLOS_ONE/NCT_CRC_Results_Final/nct_crc_subset_splits.csv',
            'model_map_override': None,  # optional dict {(exp, train_cr, fold): path} if auto-mapping fails
        },
    },
    'breakhis_rd_csv': f'{DRIVE}/BreakHis_Compressed/compression_metrics.csv',
}

# Manuscript Tables 5-7 (mean accuracy %, order CR1..CR1000) — used only to confirm the files on disk are the reported runs
MANUSCRIPT = {
    ('BreakHis', 'adaptation'): [85.61, 83.05, 84.49, 84.27, 82.43, 80.61, 78.63],
    ('BreakHis', 'robustness'): [82.81, 82.68, 82.47, 81.89, 80.97, 74.23, 61.48],
    ('BreakHis', 'cat'):        [86.31, 86.18, 86.11, 86.05, 85.98, 82.34, 74.92],
    ('BACH', 'adaptation'):     [77.75, 75.50, 72.75, 67.25, 66.75, 66.00, 48.50],
    ('BACH', 'robustness'):     [78.50, 75.25, 73.00, 69.50, 65.50, 56.75, 51.50],
    ('BACH', 'cat'):            [82.50, 80.75, 79.50, 77.00, 74.25, 62.75, 53.25],
    ('NCT-CRC', 'adaptation'):  [97.95, 98.34, 97.66, 97.79, 96.68, 85.81, 74.48],
    ('NCT-CRC', 'robustness'):  [98.17, 96.68, 90.58, 70.82, 62.96, 60.85, 60.85],
    ('NCT-CRC', 'cat'):         [98.12, 98.28, 98.08, 98.02, 96.24, 61.45, 60.85],
}

OUT = CFG['out_dir']
for sub in ['', 'inventory', 'rate_distortion', 'predictions', 'tables']:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)
print('Outputs ->', OUT)

Outputs -> /content/drive/MyDrive/JPEG2000_Revision_TierA


In [3]:
# ============================================================
# Shared helpers — copied from the original experiment code so folds
# and preprocessing are reconstructed exactly.
# ============================================================
SEED = CFG['seed']

def retry(fn, *a, tries=4, wait=3, **k):
    # Google Drive sometimes throws Errno 5 on big folders; retry instead of failing.
    for i in range(tries):
        try:
            return fn(*a, **k)
        except OSError as e:
            if i == tries - 1:
                raise
            print(f'   ⚠️ {e} — retrying in {wait}s'); time.sleep(wait)

LISTING_FAILURES = []
def listdir(p, must=True):
    if not os.path.isdir(p):
        return []
    try:
        return sorted(retry(os.listdir, p))
    except OSError as e:
        LISTING_FAILURES.append((p, str(e)))
        print(f'   ❌ could not list {p}: {e}')
        if must:
            raise
        return []

# ---- original get_patient_data (BreakHis version, 2/27 notebook) ----
def load_breakhis(data_dir, cr_level):
    rows = []
    for split in ['train', 'val', 'test']:
        split_dir = os.path.join(data_dir, cr_level, split)
        for class_idx, class_name in enumerate(['benign', 'malignant']):
            class_dir = os.path.join(split_dir, class_name)
            for mag in listdir(class_dir):
                mag_dir = os.path.join(class_dir, mag)
                if not os.path.isdir(mag_dir):
                    continue
                for fn in listdir(mag_dir):
                    if fn.endswith('.png'):
                        parts = fn.replace('.png', '').split('-')
                        pid = f'{parts[1]}-{parts[2]}' if len(parts) >= 3 else 'unknown'
                        rows.append(dict(filepath=os.path.join(mag_dir, fn), filename=fn, orig_split=split,
                                         class_name=class_name, class_idx=class_idx,
                                         patient_id=pid, magnification=mag, cr_level=cr_level))
    return pd.DataFrame(rows)

# ---- original get_patient_data (BACH fix: filename = patient_id) ----
def load_bach(data_dir, cr_level):
    rows = []
    for split in ['train', 'val', 'test']:
        for class_idx, class_name in enumerate(['benign', 'malignant']):
            class_dir = os.path.join(data_dir, cr_level, split, class_name)
            for fn in listdir(class_dir):
                if fn.endswith('.png'):
                    rows.append(dict(filepath=os.path.join(class_dir, fn), filename=fn, orig_split=split,
                                     class_name=class_name, class_idx=class_idx,
                                     patient_id=fn.replace('.png', ''), magnification='N/A', cr_level=cr_level))
    return pd.DataFrame(rows)

# ---- NCT: build from the CR1_original index, remap to other CR folders ----
_NCT_INDEX = None
def nct_index(data_dir):
    global _NCT_INDEX
    if _NCT_INDEX is None:
        rows = []
        for split in ['train', 'val', 'test']:
            for class_idx, class_name in enumerate(['benign', 'malignant']):
                d = os.path.join(data_dir, 'CR1_original', split, class_name)
                for fn in listdir(d, must=False):   # train/benign (41k files) may fail on Drive; subset never uses it
                    if fn.lower().endswith('.png'):
                        rows.append(dict(filename=fn, orig_split=split, class_name=class_name,
                                         class_idx=class_idx, patient_id=fn.rsplit('.', 1)[0],
                                         tissue=fn.split('_')[0], magnification='N/A'))
        _NCT_INDEX = pd.DataFrame(rows)
    return _NCT_INDEX

def load_nct(data_dir, cr_level, subset=None):
    df = nct_index(data_dir).copy()
    if subset is not None:
        df = df[df['filename'].isin(subset)]
    df['filepath'] = [os.path.join(data_dir, cr_level, s, c, f)
                      for s, c, f in zip(df.orig_split, df.class_name, df.filename)]
    df['cr_level'] = cr_level
    return df.reset_index(drop=True)

LOADERS = {'breakhis': load_breakhis, 'bach': load_bach}

def rotating_test_folds(image_df):
    # identical to train_model_cv(): patient-level StratifiedKFold, seed 42
    pc = image_df.groupby('patient_id')['class_idx'].first().reset_index()
    patients, labels = pc['patient_id'].values, pc['class_idx'].values
    skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=SEED)
    return [set(patients[te]) for _, te in skf.split(patients, labels)]

# ---- original ChannelAttention layer (needed to load the .keras files) ----
class ChannelAttention(layers.Layer):
    def __init__(self, ratio=4, **kwargs):
        super().__init__(**kwargs); self.ratio = ratio
    def build(self, input_shape):
        channel = input_shape[-1]
        self.shared_dense1 = layers.Dense(channel // self.ratio, activation='relu')
        self.shared_dense2 = layers.Dense(channel, activation='sigmoid')
        super().build(input_shape)
    def call(self, x):
        avg_pool = tf.reduce_mean(x, axis=1, keepdims=True)
        max_pool = tf.reduce_max(x, axis=1, keepdims=True)
        avg_out = self.shared_dense2(self.shared_dense1(avg_pool))
        max_out = self.shared_dense2(self.shared_dense1(max_pool))
        return x * (avg_out + max_out)
    def get_config(self):
        c = super().get_config(); c.update({'ratio': self.ratio}); return c

def load_model(path):
    return keras.models.load_model(path, custom_objects={'ChannelAttention': ChannelAttention}, compile=False)

def read_bytes(paths, workers=16):
    def rd(p):
        return retry(lambda: open(p, 'rb').read())
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(rd, paths))

def predict_bytes(model, byte_list):
    # same preprocessing as create_dataset(): decode_png -> resize 224 (bilinear) -> /255
    @tf.autograph.experimental.do_not_convert
    def f(b):
        img = tf.image.decode_png(b, channels=3)
        img = tf.image.resize(img, CFG['image_size'])
        return tf.cast(img, tf.float32) / 255.0
    ds = (tf.data.Dataset.from_tensor_slices(tf.constant(byte_list))
          .map(f, num_parallel_calls=tf.data.AUTOTUNE).batch(CFG['batch_size']).prefetch(tf.data.AUTOTUNE))
    return model.predict(ds, verbose=0).flatten()

print('✅ helpers ready')

✅ helpers ready


## 1. Inventory — run this first and read the output
It checks that the data, models and result CSVs are where the original code put them, and searches the NCT-CRC folder for the split file and the model folder.

In [4]:
inv = {}

def count_tree(root, max_depth=4):
    # count png files below root without one giant recursive glob (avoids Drive Errno 5)
    n, stack = 0, [(root, 0)]
    while stack:
        d, depth = stack.pop()
        try:
            entries = list(retry(os.scandir, d))
        except OSError as e:
            LISTING_FAILURES.append((d, str(e))); print(f'   ❌ listing failed: {d} ({e})')
            continue
        for e in entries:
            if e.is_dir() and depth < max_depth:
                stack.append((e.path, depth + 1))
            elif e.name.lower().endswith('.png'):
                n += 1
    return n

for ds, c in CFG['datasets'].items():
    print(f'\n=== {ds} ===')
    info = {}
    print(f"data_dir: {c['data_dir']}  exists={os.path.isdir(c['data_dir'])}")
    if os.path.isdir(c['data_dir']):
        for cr in CFG['cr_levels']:
            n = count_tree(os.path.join(c['data_dir'], cr))
            info[f'n_{cr}'] = n
            print(f'   {cr:13s} {n:>7,} png')
    if c.get('model_dir') and os.path.isdir(c['model_dir']):
        models = sorted(f for f in listdir(c['model_dir']) if f.endswith('.keras'))
        info['n_models'] = len(models)
        print(f"models: {len(models)} .keras in {c['model_dir']}")
        pat = pd.Series([re.sub(r'(CR\w+?|Mixed)_fold\d', '<CR>_fold<k>', m) for m in models]).value_counts()
        print(pat.to_string())
    else:
        print(f"models: model_dir not set or missing ({c.get('model_dir')})")
    if c.get('results_dir') and os.path.isdir(c['results_dir']):
        print('results:', [f for f in listdir(c['results_dir']) if f.endswith('.csv')])
    inv[ds] = info

# ---- search the NCT project folder for split files and model folders (skips image folders) ----
print('\n=== NCT-CRC: searching for split file / models / results ===')
SKIP = set(CFG['cr_levels']) | {'train', 'val', 'test', 'benign', 'malignant'}
nct_root = os.path.dirname(CFG['datasets']['NCT-CRC']['data_dir'])
cands = {'split': [], 'model_dirs': {}, 'results': []}
for root, dirs, files in os.walk(nct_root):
    dirs[:] = [d for d in dirs if d not in SKIP and not d.startswith('CR')]
    for f in files:
        p = os.path.join(root, f)
        fl = f.lower()
        if fl.endswith(('.csv', '.json')) and any(k in fl for k in ['split', 'indices', 'subset', 'sample']):
            cands['split'].append(p)
        if fl.endswith('.keras'):
            cands['model_dirs'][root] = cands['model_dirs'].get(root, 0) + 1
        if fl.startswith('experiment') and fl.endswith('.csv'):
            cands['results'].append(p)
print('split-file candidates:'); [print('   ', p) for p in cands['split']]
print('model folders (n .keras):'); [print(f'    {k}  ({v})') for k, v in cands['model_dirs'].items()]
print('result CSVs:'); [print('   ', p) for p in cands['results']]

# auto-fill NCT paths only when there is exactly one candidate
nc = CFG['datasets']['NCT-CRC']
if nc['model_dir'] is None and len(cands['model_dirs']) == 1:
    nc['model_dir'] = next(iter(cands['model_dirs']))
if nc['results_dir'] is None and len({os.path.dirname(p) for p in cands['results']}) == 1:
    nc['results_dir'] = os.path.dirname(cands['results'][0])
if nc['split_file'] is None and len(cands['split']) == 1:
    nc['split_file'] = cands['split'][0]
print('\nNCT config now:', {k: nc[k] for k in ['model_dir', 'results_dir', 'split_file']})
print('If any is None or wrong: set it in the CONFIG cell, re-run CONFIG + helpers, skip this cell.')

print('\nListing failures so far:', len(LISTING_FAILURES))
json.dump({'inventory': inv, 'nct_candidates': cands, 'listing_failures': LISTING_FAILURES}, open(f'{OUT}/inventory/inventory.json', 'w'), indent=2)


=== BreakHis ===
data_dir: /content/drive/MyDrive/BreakHis_Split  exists=True
   CR1_original    7,910 png
   CR15            7,910 png
   CR25            7,910 png
   CR50            7,910 png
   CR100           7,910 png
   CR500           7,910 png
   CR1000          7,910 png
models: 40 .keras in /content/drive/MyDrive/BreakHis_PLOS_ONE_FINAL/models
Inception_DenseNet169_Attention_<CR>_fold<k>.keras         35
Inception_DenseNet169_Attention_Mixed<CR>_fold<k>.keras     5
results: ['experiment1_adaptation.csv', 'experiment2_robustness.csv', 'experiment3_compression_aware.csv', 'sustainability_analysis.csv', 'table1_adaptation.csv', 'table2_robustness.csv', 'table3_sustainability.csv']

=== BACH ===
data_dir: /content/drive/MyDrive/BACH_PLOS_ONE/BACH_Split  exists=True
   CR1_original      400 png
   CR15              400 png
   CR25              400 png
   CR50              400 png
   CR100             400 png
   CR500             400 png
   CR1000            400 png
models: 40 .ke

In [5]:
# ---- read the NCT split file (format unknown, so this is defensive) ----
NCT_SPLIT = None
sf = CFG['datasets']['NCT-CRC']['split_file']
if sf:
    raw = pd.read_csv(sf) if sf.endswith('.csv') else pd.json_normalize(json.load(open(sf)))
    print(raw.head()); print('columns:', list(raw.columns), '| rows:', len(raw))
    fcol = next((c for c in raw.columns if c.lower() in ('filename', 'file', 'fname', 'image')), None)
    if fcol is None:
        pcol = next((c for c in raw.columns if 'path' in c.lower()), None)
        if pcol:
            raw['filename'] = raw[pcol].astype(str).map(os.path.basename); fcol = 'filename'
    scol = next((c for c in raw.columns if c.lower() in ('split', 'partition', 'set', 'new_split', 'subset')), None)
    if fcol and scol:
        NCT_SPLIT = raw[[fcol, scol]].rename(columns={fcol: 'filename', scol: 'split'})
        NCT_SPLIT['split'] = NCT_SPLIT['split'].astype(str).str.lower()
        print('\n✅ NCT split parsed:', NCT_SPLIT['split'].value_counts().to_dict())
    else:
        print('\n❌ Could not identify filename/split columns — send me the printed head().')
else:
    print('NCT split file not set — Sections 2 (subset part), 4 (NCT) and 5 (NCT) will be skipped.')

               filename  label  split original_split original_class_folder
0  LYM_LYM-GLGIGDMQ.png      0  train           test                benign
1  TUM_TUM-SNLENFMY.png      1  train          train             malignant
2  TUM_TUM-HYRVRHNK.png      1  train          train             malignant
3  MUC_MUC-KDEYTQSP.png      0  train            val                benign
4  LYM_LYM-PWHRCEGI.png      0  train            val                benign
columns: ['filename', 'label', 'split', 'original_split', 'original_class_folder'] | rows: 7799

✅ NCT split parsed: {'train': 5462, 'test': 1170, 'val': 1167}


In [6]:
# ---- self-test: can this TensorFlow load your saved models? (one per dataset) ----
for ds, c in CFG['datasets'].items():
    d = c.get('model_dir')
    if not d or not os.path.isdir(d):
        print(f'{ds}: no model_dir, skipped'); continue
    f = sorted(x for x in listdir(d) if x.endswith('.keras'))[0]
    try:
        m = load_model(os.path.join(d, f))
        print(f'{ds}: ✅ loaded {f} | params={m.count_params():,} | custom ChannelAttention layer: '
              f'{any(isinstance(l, ChannelAttention) for l in m.layers)} | dtype policy: {m.dtype_policy.name}')
        del m; keras.backend.clear_session()
    except Exception as e:
        print(f'{ds}: ❌ could not load {f}\n   {type(e).__name__}: {str(e)[:400]}'
              '\n   -> send me this error; the fix is usually a Keras-version shim, not retraining.')

BreakHis: ✅ loaded Inception_DenseNet169_Attention_CR1000_fold1.keras | params=36,749,793 | custom ChannelAttention layer: True | dtype policy: float32
BACH: ✅ loaded Inception_DenseNet169_Attention_CR1000_fold1.keras | params=36,749,793 | custom ChannelAttention layer: True | dtype policy: float32
NCT-CRC: ✅ loaded exp1_adaptation_CR1000_fold1.keras | params=37,669,217 | custom ChannelAttention layer: False | dtype policy: float32


## 2. NCT-CRC class verification (R1-7)
Filenames keep the original tissue code (`TUM_…`, `ADI_…`). This cross-tabulates the tissue code against the binary folder label the models were trained on. It settles which grouping was actually used.

In [7]:
nct_dir = CFG['datasets']['NCT-CRC']['data_dir']
# Why was the pool 36,579? Original processing wrote 41,343 train/benign images (NCT_CRC_Dataset_Processing.ipynb).
print('Direct count of train/benign folders (original processing wrote 41,343 per CR):')
for cr in CFG['cr_levels']:
    d = os.path.join(nct_dir, cr, 'train', 'benign')
    try:
        n = sum(1 for _ in retry(os.scandir, d)) if os.path.isdir(d) else 'folder missing'
    except OSError as e:
        n = f'LISTING FAILED: {e}'
    print(f'   {cr:13s} {n}')
print()
idx = nct_index(nct_dir)
print(f'CR1_original pool: {len(idx):,} images')
print('\nPool by original folder split x label:')
print(pd.crosstab(idx['orig_split'], idx['class_name'], margins=True))
ct_pool = pd.crosstab(idx['tissue'], idx['class_name'], margins=True)
print('\nTissue code x binary label — POOL:'); print(ct_pool)
ct_pool.to_csv(f'{OUT}/tables/nct_tissue_by_label_pool.csv')

if NCT_SPLIT is not None:
    sub = idx.merge(NCT_SPLIT, on='filename', how='inner')
    print(f'\nSubset matched: {len(sub):,} of {len(NCT_SPLIT):,} split-file rows')
    ct_sub = pd.crosstab([sub['split'], sub['tissue']], sub['class_name'], margins=True)
    print('\nTissue x label — SUBSET by experiment split:'); print(ct_sub)
    ct_sub.to_csv(f'{OUT}/tables/nct_tissue_by_label_subset.csv')

pos = sorted(idx.loc[idx.class_idx == 1, 'tissue'].unique())
neg = sorted(idx.loc[idx.class_idx == 0, 'tissue'].unique())
print(f'\nPOSITIVE (label 1) tissue codes: {pos}')
print(f'NEGATIVE (label 0) tissue codes: {neg}')
print('Manuscript claims positive = TUM, LYM, MUC, DEB; negative = ADI, BACK, MUS, NORM, STR')

Direct count of train/benign folders (original processing wrote 41,343 per CR):
   CR1_original  41343
   CR15          41343
   CR25          41343
   CR50          41343
   CR100         41343
   CR500         41343
   CR1000        41343

CR1_original pool: 77,922 images

Pool by original folder split x label:
class_name  benign  malignant    All
orig_split                          
test         12721       2864  15585
train        41343       9305  50648
val           9541       2148  11689
All          63605      14317  77922

Tissue code x binary label — POOL:
class_name  benign  malignant    All
tissue                              
ADI          10407          0  10407
LYM          11557          0  11557
MUC           8896          0   8896
MUS          13536          0  13536
NORM          8763          0   8763
STR          10446          0  10446
TUM              0      14317  14317
All          63605      14317  77922

Subset matched: 7,799 of 7,799 split-file rows

Tissue x

## 3. BreakHis: find the 7,910th image (R1 minor)

In [8]:
bdir = CFG['datasets']['BreakHis']['data_dir']
b = load_breakhis(bdir, 'CR1_original')
b['subtype'] = b['filename'].str.split('-').str[0].str.replace('SOB_', '', regex=False)
official = {'B_A': 444, 'B_F': 1014, 'B_PT': 453, 'B_TA': 569, 'M_DC': 3451, 'M_LC': 626, 'M_MC': 792, 'M_PC': 560}
cnt = b['subtype'].value_counts()
comp = pd.DataFrame({'found': cnt, 'official': pd.Series(official)}).fillna(0).astype(int)
comp['diff'] = comp['found'] - comp['official']
print(f'Total images: {len(b):,}  patients: {b.patient_id.nunique()}'); print(comp)

dups_name = b[b.duplicated('filename', keep=False)].sort_values('filename')
print(f'\nDuplicate filenames: {len(dups_name)}'); print(dups_name[['filename', 'orig_split', 'magnification']])

def md5(p):
    h = hashlib.md5(); h.update(retry(lambda: open(p, 'rb').read())); return h.hexdigest()
with ThreadPoolExecutor(16) as ex:
    b['md5'] = list(ex.map(md5, b['filepath']))
dups_md5 = b[b.duplicated('md5', keep=False)].sort_values('md5')
print(f'\nByte-identical files: {len(dups_md5)}'); print(dups_md5[['filename', 'subtype', 'patient_id', 'magnification']])

# patients that contain more than one diagnosis (affects patient-level stratification)
mixed = b.groupby('patient_id')['class_idx'].nunique()
print('\nPatients with both classes:', list(mixed[mixed > 1].index))
comp.to_csv(f'{OUT}/tables/breakhis_subtype_counts.csv'); dups_md5.to_csv(f'{OUT}/tables/breakhis_duplicates.csv', index=False)

Total images: 7,910  patients: 81
      found  official  diff
B_A     444       444     0
B_F    1014      1014     0
B_PT    453       453     0
B_TA    569       569     0
M_DC   3451      3451     0
M_LC    626       626     0
M_MC    793       792     1
M_PC    560       560     0

Duplicate filenames: 0
Empty DataFrame
Columns: [filename, orig_split, magnification]
Index: []

Byte-identical files: 2
                               filename subtype patient_id magnification
7353  SOB_M_MC-14-12773-200-011 (1).png    M_MC   14-12773          200X
7354      SOB_M_MC-14-12773-200-011.png    M_MC   14-12773          200X

Patients with both classes: []


## 4. Achieved compression ratios (R1-6)
* **BreakHis** was compressed at native 700×460 and the codestream size was logged at the time. It is summarised from `compression_metrics.csv`.
* **BACH and NCT-CRC**: the stored files are decoded PNGs, so their sizes say nothing about JPEG2000. Each original is re-encoded with the same Glymur call (`cratios=[CR]`) and the codestream bytes are recorded. The output is also checked for being **bit-identical** to the stored compressed PNG. If it is, the achieved ratios describe exactly the images the models saw.

In [9]:
rd = pd.read_csv(CFG['breakhis_rd_csv'])
print('BreakHis RD columns:', list(rd.columns), '| rows:', len(rd))
def q(p): return lambda s: s.quantile(p)
g = rd.groupby('compression_ratio')
bh = pd.DataFrame({
    'n': g.size(),
    'achieved_mean': g['achieved_cr'].mean(), 'achieved_sd': g['achieved_cr'].std(),
    'achieved_median': g['achieved_cr'].median(), 'achieved_p5': g['achieved_cr'].agg(q(.05)),
    'achieved_p95': g['achieved_cr'].agg(q(.95)), 'achieved_min': g['achieved_cr'].min(),
    'achieved_max': g['achieved_cr'].max(), 'bpp_mean': g['bpp'].mean(),
    'psnr_mean': g['psnr_db'].mean(), 'psnr_sd': g['psnr_db'].std(),
    'ssim_mean': g['ssim'].mean(), 'ssim_sd': g['ssim'].std(),
    'pct_within_5pct_of_target': g.apply(lambda d: 100 * np.mean(np.abs(d['achieved_cr'] / d.name - 1) <= 0.05)),
}).round(3)
bh.insert(0, 'dataset', 'BreakHis')
print(bh.to_string())
bh.to_csv(f'{OUT}/rate_distortion/breakhis_achieved_cr_summary.csv')

BreakHis RD columns: ['filename', 'class', 'subtype', 'patient_id', 'magnification', 'label', 'compression_ratio', 'achieved_cr', 'compressed_bytes', 'bpp', 'psnr_db', 'ssim'] | rows: 47460
                    dataset     n  achieved_mean  achieved_sd  achieved_median  achieved_p5  achieved_p95  achieved_min  achieved_max  bpp_mean  psnr_mean  psnr_sd  ssim_mean  ssim_sd  pct_within_5pct_of_target
compression_ratio                                                                                                                                                                                                 
15                 BreakHis  7910         15.027        0.027           15.020       14.998        15.082        14.996        15.146     1.597     39.324    3.101      0.970    0.010                    100.000
25                 BreakHis  7910         25.060        0.063           25.043       24.993        25.189        24.990        25.423     0.958     36.833    3.529      0.951   

In [10]:
WORKER_SRC = r'''
def _encode_one(task):
    import numpy as np, os, tempfile, glymur
    from PIL import Image
    from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim
    orig_path, stored, crs, fname = task
    out = []
    try:
        img = np.array(Image.open(orig_path).convert('RGB'))
    except Exception as e:
        return [dict(filename=fname, error=f'read_orig: {e}')]
    h, w, ch = img.shape
    for cr in crs:
        tmp = tempfile.mktemp(suffix='.jp2')
        try:
            glymur.Jp2k(tmp, data=img, cratios=[cr])
            nbytes = os.path.getsize(tmp)
            dec = glymur.Jp2k(tmp)[:]
            if dec.dtype != np.uint8:
                dec = np.clip(dec, 0, 255).astype(np.uint8)
            st = np.array(Image.open(stored[cr]).convert('RGB'))
            same = st.shape == dec.shape
            out.append(dict(filename=fname, target_cr=cr, raw_bytes=h * w * ch, codestream_bytes=nbytes,
                            achieved_cr=h * w * ch / nbytes, bpp=nbytes * 8 / (h * w),
                            psnr_db=psnr(img, dec, data_range=255),
                            ssim=ssim(img, dec, channel_axis=2, data_range=255),
                            stored_bit_identical=bool(same and np.array_equal(st, dec)),
                            stored_max_abs_diff=int(np.abs(st.astype(int) - dec.astype(int)).max()) if same else -1))
        except Exception as e:
            out.append(dict(filename=fname, target_cr=cr, error=str(e)))
        finally:
            if os.path.exists(tmp):
                os.remove(tmp)
    return out

'''
# written to a module so worker processes can import it (safe with TensorFlow loaded; avoids fork issues)
os.makedirs('/content/tierA_tmp' if IN_COLAB else '/tmp/tierA_tmp', exist_ok=True)
WORKER_DIR = '/content/tierA_tmp' if IN_COLAB else '/tmp/tierA_tmp'
open(os.path.join(WORKER_DIR, 'tierA_worker.py'), 'w').write(WORKER_SRC)
if WORKER_DIR not in sys.path:
    sys.path.insert(0, WORKER_DIR)
import importlib, multiprocessing as mp, tierA_worker
importlib.reload(tierA_worker)

def achieved_cr_reencode(ds_name, df_orig, data_dir, chunk=400):
    crs = [CFG['cr_value'][c] for c in CFG['cr_levels'][1:]]
    cr_dir = {v: k for k, v in CFG['cr_value'].items()}
    part = f'{OUT}/rate_distortion/{ds_name}_reencode_per_image.csv'
    done = set(pd.read_csv(part)['filename']) if os.path.exists(part) else set()
    todo = df_orig[~df_orig['filename'].isin(done)]
    print(f'{ds_name}: {len(done):,} done, {len(todo):,} to go')
    tasks = [(r.filepath,
              {cr: r.filepath.replace(f'{os.sep}CR1_original{os.sep}', f'{os.sep}{cr_dir[cr]}{os.sep}') for cr in crs},
              crs, r.filename) for r in todo.itertuples()]
    with ProcessPoolExecutor(max(1, os.cpu_count()), mp_context=mp.get_context('spawn'),
                             initializer=sys.path.insert, initargs=(0, WORKER_DIR)) as ex:
        for i in range(0, len(tasks), chunk):
            rows = [x for res in ex.map(tierA_worker._encode_one, tasks[i:i + chunk]) for x in res]
            pd.DataFrame(rows).to_csv(part, mode='a', header=not os.path.exists(part), index=False)
            print(f'   {min(i + chunk, len(tasks)):,}/{len(tasks):,}')
    res = pd.read_csv(part)
    if 'error' in res.columns and res['error'].notna().any():
        print('⚠️ errors:', res['error'].notna().sum()); print(res[res['error'].notna()].head())
        res = res[res['error'].isna()]
    g = res.groupby('target_cr')
    summ = pd.DataFrame({
        'n': g.size(), 'achieved_mean': g['achieved_cr'].mean(), 'achieved_sd': g['achieved_cr'].std(),
        'achieved_median': g['achieved_cr'].median(), 'achieved_p5': g['achieved_cr'].quantile(.05),
        'achieved_p95': g['achieved_cr'].quantile(.95), 'achieved_min': g['achieved_cr'].min(),
        'achieved_max': g['achieved_cr'].max(), 'codestream_bytes_min': g['codestream_bytes'].min(),
        'bpp_mean': g['bpp'].mean(), 'psnr_mean': g['psnr_db'].mean(), 'ssim_mean': g['ssim'].mean(),
        'pct_bit_identical_to_stored': 100 * g['stored_bit_identical'].mean(),
        'max_abs_diff_vs_stored': g['stored_max_abs_diff'].max(),
    }).round(3)
    summ.insert(0, 'dataset', ds_name)
    summ.to_csv(f'{OUT}/rate_distortion/{ds_name}_achieved_cr_summary.csv')
    print(summ.to_string())
    if summ['pct_bit_identical_to_stored'].min() < 99:
        print(f'\n⚠️ {ds_name}: re-encoding does NOT reproduce the stored images exactly. '
              f'Either the original call used other Glymur options or the OpenJPEG version differs '
              f'(now {glymur.version.openjpeg_version}). Achieved ratios are then indicative only — tell me.')
    return summ

c = CFG['datasets']['BACH']
bach_orig = load_bach(c['data_dir'], 'CR1_original')
bach_rd = achieved_cr_reencode('BACH', bach_orig, c['data_dir'])

if NCT_SPLIT is not None:
    c = CFG['datasets']['NCT-CRC']
    nct_orig = load_nct(c['data_dir'], 'CR1_original', subset=set(NCT_SPLIT['filename']))
    nct_rd = achieved_cr_reencode('NCT-CRC', nct_orig, c['data_dir'])

BACH: 0 done, 400 to go
   400/400
          dataset    n  achieved_mean  achieved_sd  achieved_median  achieved_p5  achieved_p95  achieved_min  achieved_max  codestream_bytes_min  bpp_mean  psnr_mean  ssim_mean  pct_bit_identical_to_stored  max_abs_diff_vs_stored
target_cr                                                                                                                                                                                                                            
15           BACH  400         15.115        0.131           15.068       14.984        15.387        14.975        15.618                  9638     1.588     27.871      0.865                        100.0                       0
25           BACH  400         25.259        0.334           25.147       24.943        25.973        24.930        26.675                  5643     0.950     25.738      0.798                        100.0                       0
50           BACH  400         50.809        

## 5. Re-predict from saved models (GPU) and check against the original CSVs
For every fold this reconstructs the **exact** test set the original code used and runs the saved model. It then stores one row per image with its probability.

Model files follow the original naming:
* `{name}_{CR}_fold{k}.keras` — adaptation (Experiment 1)
* `{name}_CR1_original_fold{k}.keras` — robustness (Experiment 2). Experiment 2 retrained CR1 under the **same filename** as Experiment 1, so the CR1 file now on disk is expected to be Experiment 2's model. The check below confirms or refutes this.
* `{name}_MixedCR_Mixed_fold{k}.keras` — CAT (Experiment 3)

In [11]:
PRED_DIR = f'{OUT}/predictions'

def model_path_rotating(c, exp, cr, k):
    n, d = c['model_name'], c['model_dir']
    if exp == 'cat':
        return os.path.join(d, f'{n}_MixedCR_Mixed_fold{k}.keras')
    return os.path.join(d, f'{n}_{cr}_fold{k}.keras')   # adaptation and robustness share the CR1 file

def predict_rotating(ds):
    c = CFG['datasets'][ds]; load = LOADERS[c['loader']]
    dfs = {cr: load(c['data_dir'], cr) for cr in CFG['cr_levels']}
    for cr, d in dfs.items():
        print(f'   {cr:13s} {len(d):,} images, {d.patient_id.nunique()} patients')
    # folds exactly as each original experiment computed them
    folds = {'robustness': rotating_test_folds(dfs['CR1_original']),
             'cat': rotating_test_folds(pd.concat([dfs[c_] for c_ in CFG['mixed_cr_levels']], ignore_index=True))}
    for cr in CFG['cr_levels']:
        folds[('adaptation', cr)] = rotating_test_folds(dfs[cr])
    same = all(folds[('adaptation', cr)] == folds['robustness'] for cr in CFG['cr_levels']) and folds['cat'] == folds['robustness']
    print(f'   folds identical across experiments/CRs: {same}')
    json.dump({f'fold{k+1}': sorted(map(str, s)) for k, s in enumerate(folds['robustness'])},
              open(f'{PRED_DIR}/{ds}_test_folds.json', 'w'), indent=1)

    for k in range(1, CFG['n_folds'] + 1):
        out = f'{PRED_DIR}/{ds}_fold{k}.csv.gz'
        if os.path.exists(out):
            print(f'   fold {k}: already done'); continue
        t0 = time.time(); rows = []
        m_rob = load_model(model_path_rotating(c, 'robustness', 'CR1_original', k))
        m_cat = load_model(model_path_rotating(c, 'cat', None, k))
        for cr in CFG['cr_levels']:
            for exp, model, tp in [('robustness', m_rob, folds['robustness'][k - 1]),
                                   ('cat', m_cat, folds['cat'][k - 1]),
                                   ('adaptation', None, folds[('adaptation', cr)][k - 1])]:
                te = dfs[cr][dfs[cr]['patient_id'].isin(tp)]
                if exp == 'adaptation':
                    if cr == 'CR1_original':
                        continue   # same file as robustness CR1; handled in the check below
                    model = load_model(model_path_rotating(c, 'adaptation', cr, k))
                probs = predict_bytes(model, read_bytes(te['filepath'].tolist()))
                rows.append(pd.DataFrame(dict(dataset=ds, exp=exp, fold=k,
                                              train_cr={'robustness': 'CR1_original', 'cat': 'Mixed'}.get(exp, cr),
                                              test_cr=cr, filename=te['filename'].values,
                                              patient_id=te['patient_id'].values,
                                              magnification=te['magnification'].values,
                                              y_true=te['class_idx'].values, prob=probs)))
                if exp == 'adaptation':
                    del model; gc.collect()
        pd.concat(rows).to_csv(out, index=False, compression='gzip')
        del m_rob, m_cat; keras.backend.clear_session(); gc.collect()
        print(f'   fold {k}: saved ({time.time() - t0:.0f}s)')

for ds in ['BreakHis', 'BACH']:
    print(f'\n=== {ds} ==='); predict_rotating(ds)


=== BreakHis ===
   CR1_original  7,910 images, 81 patients
   CR15          7,910 images, 81 patients
   CR25          7,910 images, 81 patients
   CR50          7,910 images, 81 patients
   CR100         7,910 images, 81 patients
   CR500         7,910 images, 81 patients
   CR1000        7,910 images, 81 patients
   folds identical across experiments/CRs: True
   fold 1: saved (883s)
   fold 2: saved (870s)
   fold 3: saved (989s)
   fold 4: saved (974s)
   fold 5: saved (982s)

=== BACH ===
   CR1_original  400 images, 400 patients
   CR15          400 images, 400 patients
   CR25          400 images, 400 patients
   CR50          400 images, 400 patients
   CR100         400 images, 400 patients
   CR500         400 images, 400 patients
   CR1000        400 images, 400 patients
   folds identical across experiments/CRs: True


   fold 1: saved (214s)
   fold 2: saved (287s)
   fold 3: saved (269s)
   fold 4: saved (260s)
   fold 5: saved (274s)


In [12]:
# ---- NCT-CRC: fixed test set, model filenames unknown -> map by pattern ----
def map_nct_models(model_dir):
    rx_cr = re.compile(r'(CR1_original|CR1000|CR500|CR100|CR50|CR25|CR15|CR1(?!\d))')
    rows = []
    for f in listdir(model_dir):
        if not f.endswith('.keras'):
            continue
        fl = f.lower()
        m_fold = re.search(r'fold_?(\d+)', fl)
        if re.search(r'mixed|cat|aware|exp3', fl):
            exp = 'cat'
        elif re.search(r'robust|exp2', fl):
            exp = 'robustness'
        elif re.search(r'adapt|exp1', fl):
            exp = 'adaptation'
        else:
            exp = '?'
        m_cr = rx_cr.search(f)
        cr = 'Mixed' if exp == 'cat' else (m_cr.group(1) if m_cr else '?')
        cr = 'CR1_original' if cr == 'CR1' else cr
        rows.append(dict(file=f, exp=exp, train_cr=cr, fold=int(m_fold.group(1)) if m_fold else -1))
    return pd.DataFrame(rows)

nc = CFG['datasets']['NCT-CRC']
NCT_READY = NCT_SPLIT is not None and nc['model_dir'] and os.path.isdir(nc['model_dir'])
if NCT_READY:
    mm = map_nct_models(nc['model_dir'])
    print(mm.sort_values(['exp', 'train_cr', 'fold']).to_string())
    counts = mm.groupby(['exp', 'train_cr']).size()
    expected = {('adaptation', c): 5 for c in CFG['cr_levels']} | {('robustness', 'CR1_original'): 5, ('cat', 'Mixed'): 5}
    problems = {k: counts.get(k, 0) for k, v in expected.items() if counts.get(k, 0) != v}
    if (mm['exp'] == '?').any() or (mm['fold'] < 0).any() or problems:
        NCT_READY = False
        print('\n❌ Could not map NCT models unambiguously:', problems,
              '\nSend me the table above (or fill model_map_override in CONFIG).')
    else:
        print('\n✅ NCT model mapping complete (45 files)')
else:
    print('NCT skipped (split file or model_dir missing).')

                                  file         exp      train_cr  fold
5    exp1_adaptation_CR100_fold1.keras  adaptation         CR100     1
6    exp1_adaptation_CR100_fold2.keras  adaptation         CR100     2
7    exp1_adaptation_CR100_fold3.keras  adaptation         CR100     3
8    exp1_adaptation_CR100_fold4.keras  adaptation         CR100     4
9    exp1_adaptation_CR100_fold5.keras  adaptation         CR100     5
0   exp1_adaptation_CR1000_fold1.keras  adaptation        CR1000     1
1   exp1_adaptation_CR1000_fold2.keras  adaptation        CR1000     2
2   exp1_adaptation_CR1000_fold3.keras  adaptation        CR1000     3
3   exp1_adaptation_CR1000_fold4.keras  adaptation        CR1000     4
4   exp1_adaptation_CR1000_fold5.keras  adaptation        CR1000     5
10    exp1_adaptation_CR15_fold1.keras  adaptation          CR15     1
11    exp1_adaptation_CR15_fold2.keras  adaptation          CR15     2
12    exp1_adaptation_CR15_fold3.keras  adaptation          CR15     3
13    

In [13]:
if NCT_READY:
    ds = 'NCT-CRC'
    test_files = set(NCT_SPLIT.loc[NCT_SPLIT['split'] == 'test', 'filename'])
    print(f'fixed test set: {len(test_files):,} images (expected 1,170)')
    tests = {cr: load_nct(nc['data_dir'], cr, subset=test_files) for cr in CFG['cr_levels']}
    byts = {}
    for k in range(1, CFG['n_folds'] + 1):
        out = f'{PRED_DIR}/{ds}_fold{k}.csv.gz'
        if os.path.exists(out):
            print(f'   fold {k}: already done'); continue
        t0 = time.time(); rows = []
        sel = lambda e, cr: mm[(mm.exp == e) & (mm.train_cr == cr) & (mm.fold == k)]['file'].iloc[0]
        m_rob = load_model(os.path.join(nc['model_dir'], sel('robustness', 'CR1_original')))
        m_cat = load_model(os.path.join(nc['model_dir'], sel('cat', 'Mixed')))
        for cr in CFG['cr_levels']:
            te = tests[cr]
            if cr not in byts:
                byts[cr] = read_bytes(te['filepath'].tolist())
            m_ad = load_model(os.path.join(nc['model_dir'], sel('adaptation', cr)))
            for exp, model in [('robustness', m_rob), ('cat', m_cat), ('adaptation', m_ad)]:
                rows.append(pd.DataFrame(dict(dataset=ds, exp=exp, fold=k,
                                              train_cr={'robustness': 'CR1_original', 'cat': 'Mixed'}.get(exp, cr),
                                              test_cr=cr, filename=te['filename'].values,
                                              patient_id=te['patient_id'].values, magnification='N/A',
                                              y_true=te['class_idx'].values, prob=predict_bytes(model, byts[cr]))))
            del m_ad; gc.collect()
        pd.concat(rows).to_csv(out, index=False, compression='gzip')
        del m_rob, m_cat; keras.backend.clear_session(); gc.collect()
        print(f'   fold {k}: saved ({time.time() - t0:.0f}s)')

fixed test set: 1,170 images (expected 1,170)
   fold 1: saved (756s)
   fold 2: saved (336s)
   fold 3: saved (334s)
   fold 4: saved (332s)
   fold 5: saved (325s)


In [14]:
# ============================================================
# CHECK: do re-predicted fold accuracies reproduce the original CSVs?
# ============================================================
def load_preds(ds):
    fs = sorted(glob.glob(f'{PRED_DIR}/{ds}_fold*.csv.gz'))
    return pd.concat([pd.read_csv(f) for f in fs]) if fs else None

def orig_csv(ds, name):
    d = CFG['datasets'][ds].get('results_dir')
    p = os.path.join(d, name) if d else None
    if not p or not os.path.exists(p):
        return None
    df = pd.read_csv(p)
    crcol = next(c for c in ['test_cr', 'cr_level', 'cr', 'CR'] if c in df.columns)
    df = df.rename(columns={crcol: 'cr'})
    df['cr'] = df['cr'].astype(str).str.strip().replace({'CR1': 'CR1_original', 'Original': 'CR1_original'})
    return df[['cr', 'fold', 'accuracy']]

checks = []
for ds in CFG['datasets']:
    P = load_preds(ds)
    if P is None:
        continue
    P['correct'] = ((P['prob'] >= 0.5).astype(int) == P['y_true']).astype(float)
    acc = P.groupby(['exp', 'test_cr', 'fold'])['correct'].mean().reset_index()
    for exp, fname in [('adaptation', 'experiment1_adaptation.csv'), ('robustness', 'experiment2_robustness.csv'),
                       ('cat', 'experiment3_compression_aware.csv')]:
        o = orig_csv(ds, fname)
        if o is None:
            print(f'{ds}/{exp}: original CSV not found'); continue
        for cr in CFG['cr_levels']:
            use = 'robustness' if (exp == 'adaptation' and cr == 'CR1_original' and ds != 'NCT-CRC') else exp
            a = acc[(acc.exp == use) & (acc.test_cr == cr)].set_index('fold')['correct']
            b_ = o[o.cr == cr].set_index('fold')['accuracy']
            j = pd.concat([a.rename('repred'), b_.rename('orig')], axis=1).dropna()
            if len(j):
                checks.append(dict(dataset=ds, exp=exp, cr=cr, n_folds=len(j),
                                   max_abs_diff_pp=100 * (j.repred - j.orig).abs().max(),
                                   repred_mean=100 * j.repred.mean(), orig_mean=100 * j.orig.mean()))
C = pd.DataFrame(checks).round(3)
C['match'] = C['max_abs_diff_pp'] < 0.5
print(C.to_string())
C.to_csv(f'{OUT}/tables/reprediction_check.csv', index=False)
print('\nRows that do NOT match:'); print(C[~C['match']].to_string() if (~C['match']).any() else '   none ✅')
# Are the files on disk the runs reported in the manuscript?
rows = []
for (ds, exp), vals in MANUSCRIPT.items():
    sub = C[(C.dataset == ds) & (C.exp == exp)].set_index('cr')
    for cr, v in zip(CFG['cr_levels'], vals):
        if cr in sub.index:
            rows.append(dict(dataset=ds, exp=exp, cr=cr, manuscript=v, csv_on_disk=round(sub.loc[cr, 'orig_mean'], 2),
                             diff=round(sub.loc[cr, 'orig_mean'] - v, 2)))
MS = pd.DataFrame(rows)
MS.to_csv(f'{OUT}/tables/disk_vs_manuscript.csv', index=False)
bad = MS[MS['diff'].abs() > 0.06]
print('\nCSV on disk vs manuscript tables:', 'all match ✅' if bad.empty else 'MISMATCHES ⚠️')
if not bad.empty:
    print(bad.to_string())
    print('A mismatch means the files on disk come from a different run than the paper '
          '(e.g. BACH: two runs wrote to the same folder). Tell me before using Section 6.')
print('\nExpected: BreakHis/BACH adaptation-CR1 matches neither or only roughly, because the CR1 file on disk '
      'is the robustness model. If robustness-CR1 matches experiment2 exactly, the overwrite is confirmed.')

     dataset         exp            cr  n_folds  max_abs_diff_pp  repred_mean  orig_mean  match
0   BreakHis  adaptation  CR1_original        5            7.008       82.795     85.612  False
1   BreakHis  adaptation          CR15        5            0.064       83.040     83.052   True
2   BreakHis  adaptation          CR25        5            0.140       84.440     84.495   True
3   BreakHis  adaptation          CR50        5            0.000       84.275     84.275   True
4   BreakHis  adaptation         CR100        5            0.127       82.458     82.432   True
5   BreakHis  adaptation         CR500        5            0.069       80.626     80.612   True
6   BreakHis  adaptation        CR1000        5            0.000       78.631     78.631   True
7   BreakHis  robustness  CR1_original        5            0.057       82.795     82.806   True
8   BreakHis  robustness          CR15        5            0.114       82.475     82.511   True
9   BreakHis  robustness          CR25  

## 6. Full metrics and revised statistics (R1-2, R1-3)
**Design (how to describe it in the paper):**
* **Within-model comparisons** (robustness and CAT, CR1 vs CRx) use the same trained model on the same test images. Training randomness cancels, so these are the primary tests.
* **Uncertainty:** a paired bootstrap (10,000 resamples) gives the 95% CI of Δ. Resampling is by patient for BreakHis, by image for BACH, and by test image for NCT-CRC, where each image's correctness is averaged over the 5 models. P-values come from a paired sign-flip permutation test at the same level, with **Holm correction** within each dataset.
* **Adaptation comparisons** use different models per CR, so they include training noise. For these, the notebook reports both the bootstrap and the Nadeau–Bengio corrected resampled t-test on fold accuracies. That test accounts for overlapping training sets, which is R1-2's exact point. NCT-CRC has a fixed test set, so the corrected t-test does not apply there; only the bootstrap is reported.

In [15]:
def metrics(y, p, thr=0.5):
    yh = (p >= thr).astype(int)
    tp = int(((yh == 1) & (y == 1)).sum()); tn = int(((yh == 0) & (y == 0)).sum())
    fp = int(((yh == 1) & (y == 0)).sum()); fn = int(((yh == 0) & (y == 1)).sum())
    sens = tp / (tp + fn) if tp + fn else np.nan; spec = tn / (tn + fp) if tn + fp else np.nan
    prec = tp / (tp + fp) if tp + fp else np.nan
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0
    return dict(accuracy=(tp + tn) / len(y), balanced_accuracy=np.nanmean([sens, spec]), sensitivity=sens,
                specificity=spec, precision=prec, f1=f1,
                mcc=matthews_corrcoef(y, yh) if len(np.unique(yh)) > 1 else 0.0,
                auc=roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan,
                pct_pred_positive=yh.mean(), tp=tp, tn=tn, fp=fp, fn=fn, n=len(y))

ALL = {ds: load_preds(ds) for ds in CFG['datasets']}
ALL = {k: v for k, v in ALL.items() if v is not None}

rows = []
for ds, P in ALL.items():
    for (exp, cr), g in P.groupby(['exp', 'test_cr']):
        per_fold = pd.DataFrame([metrics(f.y_true.values, f.prob.values) for _, f in g.groupby('fold')])
        r = dict(dataset=ds, exp=exp, test_cr=cr)
        for m in ['accuracy', 'balanced_accuracy', 'sensitivity', 'specificity', 'precision', 'f1', 'mcc', 'auc', 'pct_pred_positive']:
            r[f'{m}_mean'] = 100 * per_fold[m].mean() if m != 'mcc' and m != 'auc' else per_fold[m].mean()
            r[f'{m}_sd'] = 100 * per_fold[m].std() if m != 'mcc' and m != 'auc' else per_fold[m].std()
        pooled = metrics(g.y_true.values, g.prob.values)  # BreakHis/BACH: each image once; NCT: 5 models x same images
        r.update({f'pooled_{k}': v for k, v in pooled.items() if k in ('tp', 'tn', 'fp', 'fn', 'n')})
        rows.append(r)
M = pd.DataFrame(rows)
order = {c: i for i, c in enumerate(CFG['cr_levels'])}
M = M.sort_values(['dataset', 'exp', 'test_cr'], key=lambda s: s.map(order) if s.name == 'test_cr' else s).round(3)
M.to_csv(f'{OUT}/tables/full_metrics_all.csv', index=False)
print('(adaptation CR1 for BreakHis/BACH is absent here because those models were overwritten; its values come from the original CSV)')
print(M[['dataset', 'exp', 'test_cr', 'accuracy_mean', 'balanced_accuracy_mean', 'sensitivity_mean',
         'specificity_mean', 'mcc_mean', 'auc_mean', 'pct_pred_positive_mean']].to_string())

(adaptation CR1 for BreakHis/BACH is absent here because those models were overwritten; its values come from the original CSV)
     dataset         exp       test_cr  accuracy_mean  balanced_accuracy_mean  sensitivity_mean  specificity_mean  mcc_mean  auc_mean  pct_pred_positive_mean
22      BACH  adaptation          CR15         75.750                  75.750            66.000            85.500     0.533     0.847                  40.250
23      BACH  adaptation          CR25         72.750                  72.750            56.500            89.000     0.491     0.856                  33.750
24      BACH  adaptation          CR50         67.250                  67.250            64.500            70.000     0.378     0.751                  47.250
20      BACH  adaptation         CR100         66.500                  66.500            62.500            70.500     0.339     0.742                  46.000
25      BACH  adaptation         CR500         65.750                  65.750      

In [16]:
rng = np.random.default_rng(SEED)
B = CFG['n_boot']

def paired_frame(P, ds, a, b):
    # a, b = (exp, test_cr); returns per-image correctness for both conditions on identical images
    ka = P[(P.exp == a[0]) & (P.test_cr == a[1])]; kb = P[(P.exp == b[0]) & (P.test_cr == b[1])]
    key = ['filename'] if CFG['datasets'][ds]['cv'] == 'fixed_test' else ['fold', 'filename']
    j = ka.merge(kb, on=key, suffixes=('_a', '_b'))
    j['ca'] = ((j.prob_a >= .5).astype(int) == j.y_true_a).astype(float)
    j['cb'] = ((j.prob_b >= .5).astype(int) == j.y_true_b).astype(float)
    if CFG['datasets'][ds]['cv'] == 'fixed_test':   # average over the 5 models per image
        j = j.groupby('filename').agg(ca=('ca', 'mean'), cb=('cb', 'mean'), y=('y_true_a', 'first'),
                                      cluster=('patient_id_a', 'first')).reset_index()
    else:
        j = j.rename(columns={'y_true_a': 'y', 'patient_id_a': 'cluster'})
    return j

def boot_compare(j):
    j = j.assign(ca_pos=j.ca * (j.y == 1), cb_pos=j.cb * (j.y == 1), pos=(j.y == 1).astype(float))
    cl = j.groupby('cluster').agg(sa=('ca', 'sum'), sb=('cb', 'sum'), n=('ca', 'size'),
                                  pa=('ca_pos', 'sum'), pb=('cb_pos', 'sum'), npos=('pos', 'sum'))
    sa, sb, n = cl.sa.values, cl.sb.values, cl.n.values
    pa, pb, npos = cl.pa.values, cl.pb.values, cl.npos.values
    na_, nb_ = sa - pa, sb - pb; nneg = n - npos
    delta = (sa.sum() - sb.sum()) / n.sum()
    bal = lambda p_, n_, w: 0.5 * ((w @ p_) / (w @ npos) + (w @ n_) / (w @ nneg))
    w1 = np.ones(len(n)); dbal = bal(pa, na_, w1) - bal(pb, nb_, w1)
    W = rng.multinomial(len(n), np.full(len(n), 1 / len(n)), size=B).astype(float)
    d_boot = (W @ (sa - sb)) / (W @ n)
    with np.errstate(invalid='ignore', divide='ignore'):
        dbal_boot = 0.5 * ((W @ pa) / (W @ npos) + (W @ na_) / (W @ nneg)) - 0.5 * ((W @ pb) / (W @ npos) + (W @ nb_) / (W @ nneg))
    S = rng.choice([-1.0, 1.0], size=(B, len(n)))
    d_perm = (S @ (sa - sb)) / n.sum()
    p = (1 + np.sum(np.abs(d_perm) >= abs(delta) - 1e-12)) / (B + 1)
    return dict(delta_acc_pp=100 * delta, ci_low=100 * np.percentile(d_boot, 2.5), ci_high=100 * np.percentile(d_boot, 97.5),
                delta_bal_acc_pp=100 * dbal, bal_ci_low=100 * np.nanpercentile(dbal_boot, 2.5),
                bal_ci_high=100 * np.nanpercentile(dbal_boot, 97.5), p_perm=p,
                n_images=int(n.sum()), n_clusters=len(n))

def corrected_t(P, ds, a, b):
    # Nadeau & Bengio (2003) corrected resampled t-test on fold accuracies (rotating CV only)
    if CFG['datasets'][ds]['cv'] != 'rotating':
        return dict(p_corrected_t=np.nan)
    acc = lambda e, c: P[(P.exp == e) & (P.test_cr == c)].assign(
        cor=lambda d: ((d.prob >= .5).astype(int) == d.y_true).astype(float)).groupby('fold')['cor'].agg(['mean', 'size'])
    A, Bb = acc(*a), acc(*b)
    d = (A['mean'] - Bb['mean']).values; k = len(d)
    n_test = A['size'].mean(); n_total = P[(P.exp == a[0]) & (P.test_cr == a[1])].shape[0]
    ratio = n_test / (n_total - n_test)
    var = np.var(d, ddof=1)
    if var == 0:
        return dict(p_corrected_t=np.nan)
    t = d.mean() / np.sqrt((1 / k + ratio) * var)
    return dict(p_corrected_t=2 * stats.t.sf(abs(t), k - 1))

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for i, idx in enumerate(o):
        run = max(run, (m - i) * p[idx]); adj[idx] = min(1, run)
    return adj

comparisons = []
for c in CFG['cr_levels'][1:]:
    comparisons += [('within-model', ('robustness', 'CR1_original'), ('robustness', c)),
                    ('within-model', ('cat', 'CR1_original'), ('cat', c))]
for c in CFG['cr_levels']:
    comparisons.append(('CAT vs robustness', ('cat', c), ('robustness', c)))
for a_, b_ in [('CR1_original', 'CR100'), ('CR1_original', 'CR500'), ('CR100', 'CR500'),
               ('CR1_original', 'CR1000'), ('CR500', 'CR1000')]:
    comparisons.append(('adaptation (between-model)', ('adaptation', a_), ('adaptation', b_)))

res = []
for ds, P in ALL.items():
    for fam, a, b in comparisons:
        if CFG['datasets'][ds]['cv'] == 'rotating' and 'adaptation' in (a[0], b[0]) and 'CR1_original' in (a[1], b[1]):
            continue   # adaptation-CR1 models were overwritten (see Section 5 check); use the original CSV instead
        j = paired_frame(P, ds, a, b)
        if len(j) == 0:
            continue
        r = dict(dataset=ds, family=fam, A=f'{a[0]}:{a[1]}', B=f'{b[0]}:{b[1]}')
        r.update(boot_compare(j))
        if fam.startswith('adaptation'):
            r.update(corrected_t(P, ds, a, b))
        res.append(r)
R = pd.DataFrame(res)
R['p_holm'] = np.nan
for ds in R.dataset.unique():
    m = R.dataset == ds
    R.loc[m, 'p_holm'] = holm(R.loc[m, 'p_perm'].values)
R = R.round(4)
R.to_csv(f'{OUT}/tables/paired_comparisons.csv', index=False)
pd.set_option('display.width', 250)
print(R.to_string())

     dataset                      family                        A                        B  delta_acc_pp   ci_low  ci_high  delta_bal_acc_pp  bal_ci_low  bal_ci_high  p_perm  n_images  n_clusters  p_corrected_t  p_holm
0   BreakHis                within-model  robustness:CR1_original          robustness:CR15        0.2781  -0.0769   0.6492            0.1588     -0.2371       0.5593  0.1633      7910          81            NaN  1.0000
1   BreakHis                within-model         cat:CR1_original                 cat:CR15       -0.0632  -0.3665   0.2408           -0.2213     -0.5772       0.1437  0.7528      7910          81            NaN  1.0000
2   BreakHis                within-model  robustness:CR1_original          robustness:CR25        0.3919  -0.0710   0.9360            0.1212     -0.2766       0.5546  0.1521      7910          81            NaN  1.0000
3   BreakHis                within-model         cat:CR1_original                 cat:CR25       -0.0506  -0.3790   0.2537  

In [17]:
# ---- adaptation CR1 comparisons from the ORIGINAL fold CSVs (models no longer on disk) ----
rows = []
for ds in ['BreakHis', 'BACH']:
    o = orig_csv(ds, 'experiment1_adaptation.csv')
    if o is None:
        continue
    piv = o.pivot(index='fold', columns='cr', values='accuracy')
    P = ALL.get(ds)
    n_test = P[(P.exp == 'robustness') & (P.test_cr == 'CR1_original')].groupby('fold').size().mean() if P is not None else np.nan
    n_tot = P[(P.exp == 'robustness') & (P.test_cr == 'CR1_original')].shape[0] if P is not None else np.nan
    ratio = n_test / (n_tot - n_test)
    for b_ in ['CR100', 'CR500', 'CR1000']:
        d = (piv['CR1_original'] - piv[b_]).values; k = len(d)
        se_naive = np.std(d, ddof=1) / np.sqrt(k); se_corr = np.sqrt((1 / k + ratio) * np.var(d, ddof=1))
        tcrit = stats.t.ppf(.975, k - 1)
        rows.append(dict(dataset=ds, comparison=f'CR1 vs {b_}', delta_pp=100 * d.mean(),
                         naive_p=2 * stats.t.sf(abs(d.mean() / se_naive), k - 1),
                         corrected_p=2 * stats.t.sf(abs(d.mean() / se_corr), k - 1),
                         corrected_ci_low=100 * (d.mean() - tcrit * se_corr), corrected_ci_high=100 * (d.mean() + tcrit * se_corr),
                         cohens_dz=d.mean() / np.std(d, ddof=1)))
A1 = pd.DataFrame(rows).round(4)
A1.to_csv(f'{OUT}/tables/adaptation_cr1_corrected_t.csv', index=False)
print(A1.to_string())
print('\nnaive_p should reproduce the manuscript Table 9 p-values (paired t on 5 folds); corrected_p is the revised value.')

    dataset     comparison  delta_pp  naive_p  corrected_p  corrected_ci_low  corrected_ci_high  cohens_dz
0  BreakHis   CR1 vs CR100    3.1796   0.1343       0.2797           -3.8878            10.2469     0.8379
1  BreakHis   CR1 vs CR500    4.9993   0.0098       0.0368            0.4994             9.4992     2.0692
2  BreakHis  CR1 vs CR1000    6.9805   0.0021       0.0092            2.8739            11.0871     3.1659
3      BACH   CR1 vs CR100   11.0000   0.0388       0.1136           -4.1238            26.1238     1.3546
4      BACH   CR1 vs CR500   11.7500   0.0198       0.0664           -1.2750            24.7750     1.6802
5      BACH  CR1 vs CR1000   29.2500   0.0004       0.0018           18.2559            40.2441     4.9552

naive_p should reproduce the manuscript Table 9 p-values (paired t on 5 folds); corrected_p is the revised value.


In [18]:
# ---- confusion matrices for NCT-CRC collapse and a one-page summary ----
for ds, P in ALL.items():
    sub = M[(M.dataset == ds)][['exp', 'test_cr', 'accuracy_mean', 'balanced_accuracy_mean', 'mcc_mean',
                                'auc_mean', 'pct_pred_positive_mean', 'pooled_tp', 'pooled_tn', 'pooled_fp', 'pooled_fn']]
    print(f'\n==== {ds} ===='); print(sub.to_string(index=False))

summary = {
    'reprediction_all_match': bool(C['match'].all()) if len(C) else None,
    'n_comparisons': int(len(R)),
    'significant_after_holm': int((R.p_holm < .05).sum()),
    'files': sorted(os.path.relpath(p, OUT) for p in glob.glob(f'{OUT}/**/*.*', recursive=True)),
}
json.dump(summary, open(f'{OUT}/tierA_summary.json', 'w'), indent=2)
print(json.dumps(summary, indent=2))
print('\nSend me: tierA_summary.json, tables/*.csv, rate_distortion/*_summary.csv, and the Section 2 printout.')


==== BreakHis ====
       exp      test_cr  accuracy_mean  balanced_accuracy_mean  mcc_mean  auc_mean  pct_pred_positive_mean  pooled_tp  pooled_tn  pooled_fp  pooled_fn
adaptation         CR15         83.040                  78.949     0.611     0.889                  76.966       5092       1449       1031        338
adaptation         CR25         84.440                  82.072     0.644     0.886                  72.013       4948       1721        759        482
adaptation         CR50         84.275                  81.966     0.646     0.897                  72.707       4972       1677        803        458
adaptation        CR100         82.458                  79.771     0.608     0.901                  73.920       4952       1541        939        478
adaptation        CR500         80.626                  78.712     0.577     0.876                  71.418       4787       1575        905        643
adaptation       CR1000         78.631                  77.522     0.540  